# **Correlation: Asset Price vs On-Chain Activity**
Analyze the relationship between weekly average prices and on-chain transaction counts from Bitquery data.

In [6]:
#%% Load Bitquery On-Chain Data
import pandas as pd
import numpy as np

# Load on-chain transaction data from Bitquery
bitcoin_tx = pd.read_pickle('files/bitquery/bitcoin_tx.pkl')
ethereum_tx = pd.read_pickle('files/bitquery/ethereum_tx.pkl')
bnb_smart_tx = pd.read_pickle('files/bitquery/bnb_smart_tx.pkl')
avalanche_tx = pd.read_pickle('files/bitquery/avalanche_tx.pkl')
ripple_tx = pd.read_pickle('files/bitquery/ripple_tx.pkl')

# Concatenate all on-chain data
onchain_df = pd.concat([bitcoin_tx, ethereum_tx, bnb_smart_tx, avalanche_tx, ripple_tx], ignore_index=True)

# Map chain names to asset symbols (matching avg_df)
chain_to_symbol = {
    'bitcoin': 'BTC',
    'ethereum': 'ETH',
    'bnb_smart': 'BNB',
    'avalanche': 'AVAX',
    'ripple': 'XRP'
}
onchain_df['symbols'] = onchain_df['chain'].map(chain_to_symbol)

# Ensure date is datetime
onchain_df['date'] = pd.to_datetime(onchain_df['date'])

print(onchain_df.info())
print(f"\nChains loaded: {onchain_df['chain'].unique()}")
print(f"Date range: {onchain_df['date'].min()} to {onchain_df['date'].max()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12746 entries, 0 to 12745
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   chain     12746 non-null  object        
 1   date      12746 non-null  datetime64[ns]
 2   tx_count  12746 non-null  int64         
 3   symbols   12746 non-null  object        
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 398.4+ KB
None

Chains loaded: ['bitcoin' 'ethereum' 'bnb_smart' 'avalanche' 'ripple']
Date range: 2018-01-01 00:00:00 to 2026-03-25 00:00:00


In [23]:
#%% Aggregate On-Chain Data to Monthly & Merge with Price Data

# Define each asset and its respective network
asset_network_pairs = {
    'BTC': 'bitcoin',
    'ETH': 'ethereum',
    'BNB': 'bnb_smart',
    'AVAX': 'avalanche',
    'XRP': 'ripple'
}

# Aggregate daily tx_count to monthly periods
onchain_df['timestamp'] = onchain_df['date'].dt.to_period('M')
monthly_tx = onchain_df.groupby(['symbols', 'timestamp'])['tx_count'].sum().reset_index()
monthly_tx.rename(columns={'tx_count': 'monthly_tx_count'}, inplace=True)

# Load avg_df (price data)
avg_df = pd.read_pickle('files/avg_df.pkl')

# Ensure timestamp periods are compatible
avg_df['timestamp'] = avg_df['timestamp'].dt.to_timestamp().dt.to_period('M')

# Average price per month per asset (since avg_df has weekly rows)
avg_df = avg_df.groupby(['symbols', 'timestamp']).agg({'average_price': 'mean'}).reset_index()

monthly_tx['timestamp'] = monthly_tx['timestamp'].astype(str)
avg_df['timestamp'] = avg_df['timestamp'].astype(str)

# Merge price data with on-chain transaction data (only matching asset↔network pairs)
merged_df = pd.merge(avg_df, monthly_tx, on=['symbols', 'timestamp'], how='inner')
merged_df = merged_df.sort_values(['symbols', 'timestamp']).reset_index(drop=True)

print("Asset ↔ Network Pairs:")
for symbol, chain in asset_network_pairs.items():
    count = len(merged_df[merged_df['symbols'] == symbol])
    print(f"  {symbol} price  ↔  {chain} transactions  ({count} months)")

print(f"\nTotal merged records: {len(merged_df)}")
merged_df.head(10)

Asset ↔ Network Pairs:
  BTC price  ↔  bitcoin transactions  (95 months)
  ETH price  ↔  ethereum transactions  (95 months)
  BNB price  ↔  bnb_smart transactions  (68 months)
  AVAX price  ↔  avalanche transactions  (67 months)
  XRP price  ↔  ripple transactions  (86 months)

Total merged records: 411


C:\Users\Matheus\AppData\Local\Temp\ipykernel_14484\4168138390.py:24: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



,symbols,timestamp,average_price,monthly_tx_count
0,AVAX,2020-09,4.0450,57
1,AVAX,2020-10,4.0425,336
2,AVAX,2020-11,3.6580,238
3,AVAX,2020-12,3.2775,18341
4,AVAX,2021-01,10.2425,12784
5,AVAX,2021-02,32.0250,356349
6,AVAX,2021-03,29.7740,579628
7,AVAX,2021-04,29.2325,570389
8,AVAX,2021-05,26.8340,1139168
9,AVAX,2021-06,13.2500,488193


In [ ]:
#%% Correlation: Each Asset Price vs Its Own Network Activity

print("=" * 70)
print("  ASSET PRICE vs OWN NETWORK TRANSACTION ACTIVITY")
print("=" * 70)

corr_results = {}

for symbol, chain in asset_network_pairs.items():
    symbol_data = merged_df[merged_df['symbols'] == symbol].copy()
    
    if len(symbol_data) < 3:
        print(f"\n⚠ {symbol} ({chain}): Not enough data ({len(symbol_data)} months)")
        continue
    
    # Pearson correlation
    pearson = symbol_data['average_price'].corr(symbol_data['monthly_tx_count'])
    
    # Spearman correlation (Pearson on ranked values)
    rank_price = symbol_data['average_price'].rank()
    rank_tx = symbol_data['monthly_tx_count'].rank()
    spearman = rank_price.corr(rank_tx)
    
    n_months = len(symbol_data)
    date_range = f"{symbol_data['timestamp'].min()} to {symbol_data['timestamp'].max()}"
    
    corr_results[symbol] = {
        'network': chain,
        'pearson': round(pearson, 4),
        'spearman': round(spearman, 4),
        'n_months': n_months
    }
    
    print(f"\n{'─' * 50}")
    print(f"  {symbol} Price  ↔  {chain.upper()} Network Transactions")
    print(f"{'─' * 50}")
    print(f"  Months analyzed: {n_months}")
    print(f"  Date range     : {date_range}")
    print(f"  Pearson corr.  : {pearson:.4f}")
    print(f"  Spearman corr. : {spearman:.4f}")

# Summary table
print(f"\n\n{'=' * 70}")
print("  SUMMARY TABLE")
print(f"{'=' * 70}")
corr_summary = pd.DataFrame(corr_results).T
corr_summary.index.name = 'symbol'
corr_summary = corr_summary.sort_values('pearson', ascending=False)
print(corr_summary.to_string())

print("\nInterpretation:")
print("  +1.0 = Strong positive (price rises with more transactions)")
print("   0.0 = No relationship")
print("  -1.0 = Strong negative (price rises with fewer transactions)")

  ASSET PRICE vs OWN NETWORK TRANSACTION ACTIVITY

──────────────────────────────────────────────────
  BTC Price  ↔  BITCOIN Network Transactions
──────────────────────────────────────────────────
  Months analyzed: 95
  Date range     : 2018-05 to 2026-03
  Pearson corr.  : 0.5146
  Spearman corr. : 0.5187

──────────────────────────────────────────────────
  ETH Price  ↔  ETHEREUM Network Transactions
──────────────────────────────────────────────────
  Months analyzed: 95
  Date range     : 2018-05 to 2026-03
  Pearson corr.  : 0.7356
  Spearman corr. : 0.8129

──────────────────────────────────────────────────
  BNB Price  ↔  BNB_SMART Network Transactions
──────────────────────────────────────────────────
  Months analyzed: 68
  Date range     : 2020-08 to 2026-03
  Pearson corr.  : 0.8183
  Spearman corr. : 0.7533

──────────────────────────────────────────────────
  AVAX Price  ↔  AVALANCHE Network Transactions
──────────────────────────────────────────────────
  Months analyze

In [30]:
#%% Plotly: Asset Price vs Network Activity (Dual-Axis Charts)

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Color palette per asset
color_map = {
    'BTC': ['#FC922F', '#FFC88A'],   # Orange
    'ETH': ['#626AFF', '#A8ADFF'],   # Blue
    'BNB': ['#FFCF3D', '#FFE78A'],   # Yellow
    'AVAX': ['#FF3A3A', '#FF8A8A'],  # Red
    'XRP': ['#DCDCDC', '#A0A0A0'],   # Grey
}

for symbol, chain in asset_network_pairs.items():
    symbol_data = merged_df[merged_df['symbols'] == symbol].copy()
    symbol_data = symbol_data.sort_values('timestamp')

    if len(symbol_data) < 3:
        continue

    x_axis = symbol_data['timestamp'].tolist()
    prices = symbol_data['average_price'].round(2).tolist()
    tx_counts = symbol_data['monthly_tx_count'].astype(int).tolist()

    pearson_val = corr_results[symbol]['pearson']
    spearman_val = corr_results[symbol]['spearman']
    colors = color_map.get(symbol, ['#888888', '#CCCCCC'])

    # customdata: [price, tx_count] for each point
    custom = list(zip(prices, tx_counts))

    hover_tpl = (
        "<b>%{x}</b><br>"
        "Price: $%{customdata[0]:,.2f}<br>"
        "Tx: %{customdata[1]:,}"
        "<extra></extra>"
    )

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # Bar: transaction count (left y-axis)
    fig.add_trace(
        go.Bar(
            x=x_axis, y=tx_counts,
            name=f"{chain.upper()} Transactions",
            marker_color=colors[1], opacity=0.5,
            customdata=custom,
            hovertemplate=hover_tpl,
        ),
        secondary_y=False,
    )

    # Line: asset price (right y-axis)
    fig.add_trace(
        go.Scatter(
            x=x_axis, y=prices,
            name=f"{symbol} Price",
            mode="lines",
            line=dict(color=colors[0], width=2),
            customdata=custom,
            hovertemplate=hover_tpl,
        ),
        secondary_y=True,
    )

    fig.update_layout(
        title=dict(
            text=f"{symbol} Price vs {chain.upper()} Network Activity"
                 f"<br><sup>Pearson: {pearson_val}  |  Spearman: {spearman_val}</sup>",
        ),
        template="plotly_dark",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        height=500,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        xaxis=dict(
            dtick="M3",
            tickformat="%Y-%m",
            rangeslider=dict(visible=True, thickness=0.08),
            showgrid=False,
        ),
        hovermode="closest",
    )

    # Left y-axis: faint grid only
    fig.update_yaxes(
        title_text="Tx Count",
        showgrid=True,
        gridcolor="rgba(255,255,255,0.08)",
        secondary_y=False,
    )
    # Right y-axis: no grid
    fig.update_yaxes(
        title_text=f"{symbol} Price (USD)",
        showgrid=False,
        secondary_y=True,
    )

    fig.show()

In [ ]:
#%% Cross-Asset Correlation Matrix Heatmap

import plotly.express as px

# Pivot: each column = one asset's price or tx count per month
pivot_price = merged_df.pivot_table(index='timestamp', columns='symbols', values='average_price')
pivot_tx = merged_df.pivot_table(index='timestamp', columns='symbols', values='monthly_tx_count')

# Rename columns to show asset↔network pairing
pivot_price.columns = [f"{c}_price" for c in pivot_price.columns]
pivot_tx.columns = [f"{asset_network_pairs[c].upper()}_tx" for c in pivot_tx.columns]

# Combine into a single wide dataframe
wide_df = pd.concat([pivot_price, pivot_tx], axis=1).dropna()

full_corr = wide_df.corr().round(3)

fig = px.imshow(
    full_corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title=f"Cross-Asset Correlation Matrix ({len(wide_df)} overlapping months)",
)

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=600,
    width=800,
)

fig.show()